# FlashVSR+ Video Upscaler (Kaggle / Colab)

One-step diffusion video super-resolution: 4x upscale of low-res videos with temporal consistency.
Models: Turbo (fast) / Base (best detail). Requires **GPU T4 x2** (Kaggle) or any 16GB+ GPU.

- Official repo: https://github.com/OpenImagingLab/FlashVSR_Plus
- Weights: https://huggingface.co/ruchangyao/FlashVSR_plus
- Output sizes must be multiples of 128. 1080p -> request 1920x1152; 1440p -> 2560x1408.

In [1]:
# Step 1: Setup
import os, subprocess, sys, time

REPO = "FlashVSR_Plus"
COMMIT = "b548a2cc2c929d3d4676f2973abe27876ec4dee7"  # pinned 16 Oct 2026

if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "init", "-q", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "add", "origin",
                    "https://github.com/OpenImagingLab/FlashVSR_Plus.git"], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "FETCH_HEAD"], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-e", ".", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "safetensors", "einops", "imageio-ffmpeg", "gradio"], check=True)
print("setup OK")

setup OK


In [2]:
# Step 2: Download Turbo weights (~7 GB)
# turbo: dit.safetensors (5.7GB) + lq_proj_in.safetensors (1.2GB) + shared TCDecoder.ckpt
import os
from huggingface_hub import hf_hub_download

REPO_ID = "ruchangyao/FlashVSR_plus"
REV = "46b79520946ac67dec94d1566490690953c7debc"  # pinned 14 Oct 2026

os.makedirs("ckpts/turbo", exist_ok=True)
for f in ["turbo/dit.safetensors", "turbo/lq_proj_in.safetensors"]:
    p = hf_hub_download(repo_id=REPO_ID, filename=f, revision=REV)
    dst = os.path.join("ckpts", os.path.basename(os.path.dirname(p)), os.path.basename(p))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        os.replace(p, dst)

p = hf_hub_download(repo_id=REPO_ID, filename="TCDecoder.ckpt", revision=REV)
if not os.path.exists("ckpts/TCDecoder.ckpt"):
    os.replace(p, "ckpts/TCDecoder.ckpt")
print("weights OK")
print(os.listdir("ckpts"), os.listdir("ckpts/turbo"))

turbo/dit.safetensors: reconstructing file:   0%|          |  0.00B / 5.68GB            

turbo/dit.safetensors: downloading bytes:           |  0.00B            

turbo/lq_proj_in.safetensors: reconstructing file:   0%|          |  0.00B / 1.15GB            

turbo/lq_proj_in.safetensors: downloading bytes:           |  0.00B            

TCDecoder.ckpt: reconstructing file:   0%|          |  0.00B /  189MB            

TCDecoder.ckpt: downloading bytes:           |  0.00B            

weights OK
['turbo', '.gitkeep', 'TCDecoder.ckpt'] ['lq_proj_in.safetensors', 'dit.safetensors']


In [ ]:
# Step 3: choose input video
# Option A: Kaggle dataset attached (Add Input). Option B: upload below.
import glob, shutil, os

vids = glob.glob("/kaggle/input/**/*.mp4", recursive=True) + \
        glob.glob("/kaggle/input/**/*.mkv", recursive=True)
if vids:
    INPUT = vids[0]
    print("using attached:", INPUT)
else:
    from IPython.display import display
    from ipywidgets import FileUpload
    up = FileUpload(accept="video/*", multiple=False)
    display(up)
    # wait for upload
    import time
    for _ in range(300):
        if up.value:
            break
        time.sleep(0.5)
    fn = next(iter(up.value))
    INPUT = os.path.join("/kaggle/working", fn)
    with open(INPUT, "wb") as f:
        f.write(up.value[fn]["content"])
    print("uploaded:", INPUT)

# inspect dimensions
import subprocess
out = subprocess.check_output(["ffprobe", "-v", "error", "-select_streams", "v:0",
                               "-show_entries", "stream=width,height",
                               "-of", "csv=p=0", INPUT], text=True).strip()
W, H = map(int, out.split(","))
print("input", W, "x", H)

FileUpload(value={}, accept='video/*', description='Upload')

In [ ]:
# Step 4: Run upscale
# Pick output size (multiples of 128). Turbo mode = fast.
OUT_W = 1920   # e.g. 1280 (720p), 1920 (1080p), 2560 (1440p)
OUT_H = 1152   # must be divisible by 128
MODE = "turbo"  # or "base"

cmd = [sys.executable, "-m", "inference.run_stream",
       "--input", INPUT,
       "--mode", MODE,
       "--output", "/kaggle/working/output.mp4",
       "--width", str(OUT_W), "--height", str(OUT_H),
       "--decode-every", "28",
       "--hardware-profile", "generic"]   # SDPA fallback for T4

print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("DONE -> /kaggle/working/output.mp4")

In [ ]:
# Download result
from IPython.display import Video
Video("/kaggle/working/output.mp4", width=640)